# Phase 0 - Data Ingestion

This notebook converts the raw Reddit dumps collected with the Arctic Shift tool into a set of compressed, column-selected Parquet files, one per subreddit, that the rest of the pipeline consumes. It also documents and corrects a silent download-truncation problem discovered on three subreddits during ingestion.

**Inputs:** raw `*_posts.jsonl` and `*_comments.jsonl` dumps under `RAW`.
**Outputs:** per-subreddit Parquet files under `PARTS`, capped and month-stratified.

The working directory is set to the project root so that all relative paths resolve; adjust the path in the setup cell if running elsewhere

In [1]:
import json, os, glob
import pandas as pd
from tqdm import tqdm

In [2]:
import json, os, glob
import pandas as pd
from tqdm import tqdm

os.chdir("..")          # run from project root, not notebooks/

RAW = "data/raw"
PARTS = "data/interim/parts"
os.makedirs(PARTS, exist_ok=True)

CHUNK = 100_000         # rows per in-memory batch

POST_COLS = ["id","author","subreddit","created_utc",
             "title","selftext","score","num_comments","upvote_ratio",
             "link_flair_text","permalink","over_18","distinguished","stickied"]

COMM_COLS = ["id","author","subreddit","created_utc",
             "body","score","controversiality","link_id","parent_id",
             "is_submitter","permalink","distinguished","stickied"]

## 1. Conversion helper

`jsonl_to_parquet` streams a JSONL dump line by line and writes a compressed (zstd) Parquet file containing only the columns needed downstream. Streaming keeps memory bounded on multi-gigabyte dumps; malformed lines are counted and skipped rather than aborting the run; and an empty or fully-malformed file yields an empty frame rather than raising.

In [4]:
def jsonl_to_parquet(path, cols, out_path):
    if os.path.exists(out_path):
        print("skip", os.path.basename(out_path))
        return
    frames, rows, bad = [], [], 0
    with open(path, encoding="utf-8") as f:
        for line in tqdm(f, desc=os.path.basename(path)):
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                bad += 1
                continue
            rows.append({c: d.get(c) for c in cols})
            if len(rows) >= CHUNK:
                frames.append(pd.DataFrame(rows))
                rows = []
    if rows:
        frames.append(pd.DataFrame(rows))
    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(out_path, compression="zstd")
    if bad:
        print("  malformed lines:", bad)
    return len(df)

## 2. Convert all dumps to Parquet

Every posts and comments dump in `RAW` is converted. Files already converted are skipped, so the cell is safe to re-run.

In [5]:
for p in sorted(glob.glob(f"{RAW}/*_posts.jsonl")):
    out = f"{PARTS}/{os.path.basename(p).replace('.jsonl','.parquet')}"
    jsonl_to_parquet(p, POST_COLS, out)

for p in sorted(glob.glob(f"{RAW}/*_comments.jsonl")):
    out = f"{PARTS}/{os.path.basename(p).replace('.jsonl','.parquet')}"
    jsonl_to_parquet(p, COMM_COLS, out)

skip r_AKB48_posts.parquet
skip r_StrayKids_posts.parquet
skip r_TWICEsnark_posts.parquet
skip r_bangtan_posts.parquet
skip r_blackpink_posts.parquet
skip r_enhypen_posts.parquet
skip r_jihyo_posts.parquet
skip r_jpop_posts.parquet
skip r_kpop_uncensored_posts.parquet
skip r_kpopcollections_posts.parquet
skip r_kpopharshopinions_posts.parquet
skip r_kpoprants_posts.parquet
skip r_kpoptrulyuncensored_posts.parquet
skip r_nayeon_posts.parquet
skip r_nogizaka46_posts.parquet
skip r_popculture_posts.parquet
skip r_sakurazaka46_posts.parquet
skip r_twice_posts.parquet
skip r_unpopularkpopopinions_posts.parquet
skip r_AKB48_comments.parquet
skip r_StrayKids_comments.parquet
skip r_TWICEsnark_comments.parquet
skip r_blackpink_comments.parquet
skip r_enhypen_comments.parquet
skip r_jihyo_comments.parquet
skip r_jpop_comments.parquet
skip r_kpop_uncensored_comments.parquet
skip r_kpopcollections_comments.parquet
skip r_kpopharshopinions_comments.parquet
skip r_kpoprants_comments.parquet
skip r_

## 3. Inventory and per-subreddit audit

The cells below list the files the conversion loop will process and summarise the row counts and date spans per subreddit. This audit is what surfaced the truncation problem addressed in the next section.

In [4]:
import glob, os
posts = sorted(glob.glob(f"{RAW}/*_posts.jsonl"))
comms = sorted(glob.glob(f"{RAW}/*_comments.jsonl"))
print("comment files the loop will process:")
for p in comms:
    print("  ", os.path.basename(p))

comment files the loop will process:
   r_AKB48_comments.jsonl
   r_StrayKids_comments.jsonl
   r_TWICEsnark_comments.jsonl
   r_blackpink_comments.jsonl
   r_enhypen_comments.jsonl
   r_jihyo_comments.jsonl
   r_jpop_comments.jsonl
   r_kpop_uncensored_comments.jsonl
   r_kpopcollections_comments.jsonl
   r_kpopharshopinions_comments.jsonl
   r_kpoprants_comments.jsonl
   r_kpoptrulyuncensored_comments.jsonl
   r_nayeon_comments.jsonl
   r_nogizaka46_comments.jsonl
   r_popculture_comments.jsonl
   r_sakurazaka46_comments.jsonl
   r_twice_comments.jsonl
   r_unpopularkpopopinions_comments.jsonl


In [5]:
rows = []
for p in sorted(glob.glob(f"{PARTS}/*.parquet")):
    name = os.path.basename(p).replace(".parquet","")
    d = pd.read_parquet(p, columns=["created_utc","author","subreddit"])
    ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
    rows.append({
        "file": name,
        "subreddit": d.subreddit.iloc[0],
        "kind": "comments" if "comments" in name else "posts",
        "n": len(d),
        "users": d.author.nunique(),
        "first": ts.min().date(),
        "last": ts.max().date(),
        "days": (ts.max() - ts.min()).days,
    })
    del d

man = pd.DataFrame(rows).sort_values(["subreddit","kind"])
man.to_csv("data/interim/manifest.csv", index=False)
man

,file,subreddit,kind,n,users,first,last,days
0,capped_r_AKB48_comments,AKB48,comments,83567,4755,2020-01-01,2026-07-08,2380
1,capped_r_AKB48_comments_NEW,AKB48,comments,112289,6402,2011-12-15,2026-07-22,5333
38,r_AKB48_comments,AKB48,comments,112289,6402,2011-12-15,2026-07-22,5333
39,r_AKB48_comments_NEW,AKB48,comments,112289,6402,2011-12-15,2026-07-22,5333
40,r_AKB48_posts,AKB48,posts,22036,2647,2011-03-22,2026-07-22,5601
...,...,...,...,...,...,...,...,...
74,r_twice_posts,twice,posts,129587,20857,2015-06-15,2026-07-06,4038
36,capped_r_unpopularkpopopinions_comments,unpopularkpopopinions,comments,299951,32252,2018-03-13,2026-07-22,3053
75,r_unpopularkpopopinions_comments,unpopularkpopopinions,comments,1024916,51795,2018-03-13,2026-07-22,3053
37,capped_r_unpopularkpopopinions_posts,unpopularkpopopinions,posts,57404,14830,2018-03-13,2026-07-22,3053


## 4. Silent truncation fix: r/bangtan

The audit revealed that the r/bangtan comments dump had been silently truncated on download, capturing only part of the community's history. The cells below re-fetch the missing coverage in date slices, verify the slice files, and consolidate them.

*Note: the cells that read raw slice files depend on the raw dumps, which are not included in the submitted repository for size and privacy reasons; their saved error output documents the re-fetch step rather than a defect in the pipeline.*

In [6]:
for f in sorted(glob.glob(f"{PARTS}/*bangtan_comments*.parquet")):
    print(os.path.basename(f), os.path.getsize(f)//1024, "KB")

capped_r_bangtan_comments.parquet 36118 KB
r_bangtan_comments.parquet 300542 KB


In [22]:
import os, glob
os.makedirs(f"{PARTS}/slices", exist_ok=True)
for f in glob.glob(f"{PARTS}/r_bangtan_comments_*.parquet"):
    os.rename(f, f"{PARTS}/slices/{os.path.basename(f)}")
print("moved; remaining bangtan:",
      [os.path.basename(x) for x in glob.glob(f"{PARTS}/*bangtan_comments*.parquet")])

moved; remaining bangtan: []


In [23]:
for f in glob.glob(f"{PARTS}/capped_r_bangtan_comments.parquet"):
    os.remove(f)

## 5. Capping and month-stratified sampling

Each subreddit is capped at 300,000 items. Where a community exceeds the cap, a month-stratified sample is taken so that the temporal spread of the community is preserved rather than biasing toward its most active period.

In [8]:
CAP = 300_000

for p in sorted(glob.glob(f"{PARTS}/*.parquet")):
    name = os.path.basename(p)
    if name.startswith("capped_"):
        continue
    out = f"{PARTS}/capped_{name}"
    if os.path.exists(out):
        continue

    d = pd.read_parquet(p)
    if len(d) > CAP:
        m = pd.to_datetime(d.created_utc, unit="s").dt.to_period("M")
        frac = CAP / len(d)
        idx = (d.groupby(m, group_keys=False)
                 .apply(lambda g: g.sample(max(int(len(g) * frac), 1),
                                           random_state=0))
                 .index)
        d = d.loc[idx]
    d.to_parquet(out, compression="zstd")
    print(name, "->", len(d))
    del d

r_AKB48_posts.parquet -> 22036


## 6. Silent truncation fix: r/kpop_uncensored

A second subreddit, r/kpop_uncensored, was found to be truncated. It is re-fetched, merged with the existing partial data, de-duplicated, and its month distribution checked. As above, the raw-file reads depend on dumps outside the submitted repo.

In [9]:
p = f"{RAW}/r_kpop_uncensored_comments_2505.jsonl"
out = f"{PARTS}/r_kpop_uncensored_comments_2505.parquet"
n = jsonl_to_parquet(p, COMM_COLS, out)
print("wrote", n, "rows")

r_kpop_uncensored_comments_2505.jsonl: 711611it [00:12, 55731.41it/s]


wrote 711611 rows


In [ ]:
path = f"{PARTS}/r_kpop_uncensored_comments.parquet"
d = pd.concat([pd.read_parquet(path), pd.read_parquet(f"{PARTS}/r_kpop_uncensored_comments_2505.parquet")])

print(f"Before: {len(d)} | Dupes: {d.id.duplicated().sum()}")

d = d.drop_duplicates("id", ignore_index=True)
d.to_parquet(path, compression="zstd")

ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
print(f"{ts.min().date()} → {ts.max().date()}")

before: 2505355 dupes: 711611
2023-08-21 → 2026-07-22


In [11]:
old = pd.read_parquet(f"{PARTS}/r_kpop_uncensored_comments.parquet", columns=["created_utc"])
ts = pd.to_datetime(old.created_utc, unit="s", utc=True)
print(ts.dt.to_period("M").value_counts().sort_index())

created_utc
2023-08      199
2023-09     7094
2023-10    15700
2023-11    34628
2023-12    39505
2024-01    43190
2024-02    60761
2024-03    60726
2024-04    48323
2024-05    54157
2024-06    56417
2024-07    60945
2024-08    60488
2024-09    67152
2024-10    51326
2024-11    50588
2024-12    66952
2025-01    67496
2025-02    74569
2025-03    98116
2025-04    63797
2025-05    69187
2025-06    81832
2025-07    72794
2025-08    55601
2025-09    48295
2025-10    45708
2025-11    41929
2025-12    53027
2026-01    51228
2026-02    34026
2026-03    45375
2026-04    41263
2026-05    36908
2026-06    20610
2026-07    13832
Freq: M, Name: count, dtype: int64


/tmp/ipykernel_689/1502247694.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(ts.dt.to_period("M").value_counts().sort_index())


## 7. Silent truncation fix: r/popculture (control)

The r/popculture control community was the third affected subreddit. It is re-fetched in date slices, converted, consolidated and de-duplicated. The control community is used later only for domain-adaptation contrast, not for the behavioural typology.

In [12]:
d = pd.read_parquet(f"{PARTS}/r_popculture_comments.parquet", columns=["created_utc"])
ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
print(ts.dt.to_period("M").value_counts().sort_index())


created_utc
2020-01        8
2020-02        8
2020-03       11
2020-04        7
2020-05        1
           ...  
2026-03     9611
2026-04     9326
2026-05     6985
2026-06    29125
2026-07    38845
Freq: M, Name: count, Length: 77, dtype: int64


/tmp/ipykernel_689/1696957870.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(ts.dt.to_period("M").value_counts().sort_index())


In [13]:
for f in sorted(glob.glob(f"{RAW}/*popculture*")):
    print(repr(os.path.basename(f)))

'r_popculture_comments.jsonl'
'r_popculture_comments.jsonl.crswap'
'r_popculture_comments_2022.jsonl'
'r_popculture_comments_2224.jsonl'
'r_popculture_comments_2425.jsonl'
'r_popculture_comments_2526.jsonl'
'r_popculture_posts.jsonl'


In [14]:
slices = sorted(glob.glob(f"{RAW}/r_popculture_comments_*.jsonl"))
print("converting", len(slices))
for p in slices:
    out = f"{PARTS}/{os.path.basename(p).replace('.jsonl','.parquet')}"
    try:
        n = jsonl_to_parquet(p, COMM_COLS, out)
        print("OK", os.path.basename(out), "->", n)
    except Exception as e:
        print("FAILED", os.path.basename(p), "->", type(e).__name__, str(e)[:150])

converting 4


r_popculture_comments_2022.jsonl: 212it [00:00, 48098.26it/s]


OK r_popculture_comments_2022.parquet -> 212


r_popculture_comments_2224.jsonl: 250it [00:00, 53181.32it/s]


OK r_popculture_comments_2224.parquet -> 250


r_popculture_comments_2425.jsonl: 101291it [00:01, 59633.28it/s]


OK r_popculture_comments_2425.parquet -> 101291


r_popculture_comments_2526.jsonl: 864115it [00:14, 59575.60it/s]


OK r_popculture_comments_2526.parquet -> 864115


In [15]:
parts = sorted(glob.glob(f"{PARTS}/r_popculture_comments*.parquet"))
print(parts)
d = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
print("before:", len(d), "dupes:", d.id.duplicated().sum())
d = d.drop_duplicates(subset="id")
d.to_parquet(f"{PARTS}/r_popculture_comments.parquet", compression="zstd")

ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
print(len(d), ts.min().date(), "→", ts.max().date())

['data/interim/parts/r_popculture_comments.parquet', 'data/interim/parts/r_popculture_comments_2022.parquet', 'data/interim/parts/r_popculture_comments_2224.parquet', 'data/interim/parts/r_popculture_comments_2425.parquet', 'data/interim/parts/r_popculture_comments_2526.parquet']
before: 1931742 dupes: 965868
965874 2020-01-06 → 2026-07-22


## 8. Re-fetch: r/AKB48 historical coverage

The J-pop community r/AKB48 was re-fetched to extend its historical coverage (2011-2019), improving representation of the J-pop sub-population.

In [16]:
n = jsonl_to_parquet(f"{RAW}/r_AKB48_comments.jsonl", COMM_COLS,
                     f"{PARTS}/r_AKB48_comments_NEW.parquet")
d = pd.read_parquet(f"{PARTS}/r_AKB48_comments_NEW.parquet", columns=["created_utc"])
ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
print(len(d), ts.min().date(), "→", ts.max().date())

skip r_AKB48_comments_NEW.parquet
112289 2011-12-15 → 2026-07-22


## 9. Additional subreddits

A further set of member- and topic-specific subreddits is converted and capped, bringing the corpus to its final twenty communities spanning K-pop, J-pop, general discussion and snark/critique spaces.

In [ ]:
subs = ["enhypen","jihyo","kpoprants","nayeon","nogizaka46",
       "sakurazaka46","StrayKids","unpopularkpopopinions"]

for sub in subs:
    for kind, cols in (("posts", POST_COLS), ("comments", COMM_COLS)):
        try:
            n = jsonl_to_parquet(f"{RAW}/r_{sub}_{kind}.jsonl", cols, f"{PARTS}/r_{sub}_{kind}.parquet")
            print(f"OK {sub} {kind} -> {n}")
        except Exception as e:
            print(f"FAIL {sub} {kind} -> {type(e).__name__}: {str(e)[:120]}")

skip r_enhypen_posts.parquet
OK enhypen posts -> None
skip r_enhypen_comments.parquet
OK enhypen comments -> None
skip r_jihyo_posts.parquet
OK jihyo posts -> None
skip r_jihyo_comments.parquet
OK jihyo comments -> None
skip r_kpoprants_posts.parquet
OK kpoprants posts -> None
skip r_kpoprants_comments.parquet
OK kpoprants comments -> None
skip r_nayeon_posts.parquet
OK nayeon posts -> None
skip r_nayeon_comments.parquet
OK nayeon comments -> None
skip r_nogizaka46_posts.parquet
OK nogizaka46 posts -> None
skip r_nogizaka46_comments.parquet
OK nogizaka46 comments -> None
skip r_sakurazaka46_posts.parquet
OK sakurazaka46 posts -> None
skip r_sakurazaka46_comments.parquet
OK sakurazaka46 comments -> None
skip r_StrayKids_posts.parquet
OK StrayKids posts -> None
skip r_StrayKids_comments.parquet
OK StrayKids comments -> None
skip r_unpopularkpopopinions_posts.parquet
OK unpopularkpopopinions posts -> None
skip r_unpopularkpopopinions_comments.parquet
OK unpopularkpopopinions comments -> N

In [ ]:
os.makedirs(f"{PARTS}/slices", exist_ok=True)
for f in glob.glob(f"{PARTS}/r_popculture_comments_*.parquet"):
    os.rename(f, f"{PARTS}/slices/{os.path.basename(f)}")
os.remove(f"{PARTS}/capped_r_popculture_comments.parquet")

## 10. Housekeeping and final verification

Windows `Zone.Identifier` artefacts are removed, the truncation-fixed subreddits are re-checked for row count and date span, and the final capped corpus is summarised.

In [37]:
for f in glob.glob(f"{RAW}/*Zone.Identifier*"):
    os.remove(f)

In [38]:
import glob, os
for f in sorted(glob.glob(f"{PARTS}/*kpop_uncensored_comments*.parquet")):
    print(os.path.basename(f), os.path.getsize(f)//1024, "KB")

In [ ]:
d = pd.read_parquet(f"{PARTS}/r_kpop_uncensored_comments.parquet", columns=["created_utc"])
ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
print(len(d), "rows")
print(ts.min().date(), "→", ts.max().date())
print(ts.dt.to_period("M").value_counts().sort_index().head(6))
print("...")
print(ts.dt.to_period("M").value_counts().sort_index().tail(6))

In [6]:
import glob, os
rows = []
for p in sorted(glob.glob(f"{PARTS}/capped_*.parquet")):
    d = pd.read_parquet(p, columns=["created_utc", "subreddit"])
    ts = pd.to_datetime(d.created_utc, unit="s", utc=True)
    rows.append({
        "file": os.path.basename(p),
        "n": len(d),
        "first": ts.min().date(),
        "last": ts.max().date(),
    })
    del d
pd.DataFrame(rows)

,file,n,first,last
0,capped_r_AKB48_comments.parquet,112289,2011-12-15,2026-07-22
1,capped_r_AKB48_posts.parquet,22036,2011-03-22,2026-07-22
2,capped_r_StrayKids_comments.parquet,299948,2017-12-23,2026-07-23
3,capped_r_StrayKids_posts.parquet,57095,2017-12-04,2026-07-23
4,capped_r_TWICEsnark_comments.parquet,6461,2025-08-06,2026-07-06
5,capped_r_TWICEsnark_posts.parquet,373,2025-07-18,2026-06-30
6,capped_r_bangtan_comments.parquet,299945,2017-01-01,2026-07-05
7,capped_r_bangtan_posts.parquet,140336,2014-01-12,2026-07-06
8,capped_r_blackpink_comments.parquet,299957,2019-06-28,2026-07-07
9,capped_r_blackpink_posts.parquet,75099,2019-06-28,2026-07-06
